# IAC Training on TA-RWARE — Google Colab (T4 GPU)

**Project:** Scalable MARL for Warehouse Logistics (Krnjaic et al., 2023 — arXiv 2212.11498v3)  
**Algorithm:** IAC — Independent Actor-Critic (paper Table I, GTP section)  
**Environment:** Small GTP — `tarware-medium-8agvs-4pickers-partialobs-v1`  
**Framework:** [EPyMARL v2](https://github.com/uoe-agents/epymarl) (paper authors' official training framework)

> **Before running:** Go to `Runtime > Change runtime type > T4 GPU`

---

## What this notebook does

Our previous experiments (Exp 1–4) used a rule-based heuristic (CTA). This notebook trains a real
neuronal network using Reinforcement Learning — specifically **IAC**, which is the simplest
algorithm from the paper.

| Method | Small GTP Pick Rate |
|--------|--------------------|
| Random baseline (our Exp 2) | 9.63 ± 0.12 |
| CTA heuristic (our Exp 1) | 52.16 ± 0.54 |
| **IAC neural net (this notebook)** | **~40–65 (3000 ep training)** |
| IAC — paper full training (10k ep) | 65.2 ± 0.5 |
| HIAC — paper best (10k ep) | 66.7 ± 0.3 |

Pick rate = order-lines delivered per hour. Higher is better.

---

## IAC Algorithm (Independent Actor-Critic)

Each agent learns its own **Actor** (policy network) and **Critic** (value network):

```
Observation  ─►  [FC 64] ─► [FC 64]  ─►  Actor head  ─►  Action probabilities
                                      ─►  Critic head ─►  Value estimate
```

- **12 independent agents** (8 AGVs + 4 pickers), each with their own network weights
- Training uses **Generalized Advantage Estimation (GAE)** with γ = 0.99
- No weight sharing between agents (unlike SNAC) — each agent learns independently
- This matches the paper's IAC configuration exactly

## Step 0: Mount Google Drive

We save training results to Drive so they persist after the Colab session ends.
(Training takes 1–2 hours — Drive prevents losing results if the session disconnects.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/tarware_iac_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

## Step 1: Install EPyMARL and TA-RWARE

We install two repos:
- **TA-RWARE** — the warehouse simulation environment (same code as our experiments)
- **EPyMARL** — the official MARL training framework used by the paper authors

In [ ]:
import subprocess, sys, os

EPYMARL_DIR = '/content/epymarl'
PIP = f'"{sys.executable}" -m pip'

def run(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if result.returncode != 0:
        print(f'  STDERR: {result.stderr[-1000:]}')
    else:
        last = result.stdout.strip().split('\n')[-1] if result.stdout.strip() else 'OK'
        print(f'  {last}')
    return result.returncode

print('=== Installing TA-RWARE ===')
run('git clone --quiet https://github.com/uoe-agents/task-assignment-robotic-warehouse tarware_repo')
run(f'{PIP} install -q -e tarware_repo/')

print('\n=== Installing EPyMARL ===')
run('git clone --quiet https://github.com/uoe-agents/epymarl')
run(f'{PIP} install -q -r epymarl/requirements.txt')

print('\n=== Installing extras ===')
run(f'{PIP} install -q pyastar2d sacred pymongo')

ENVS_DIR = f'{EPYMARL_DIR}/src/envs'

# ── Patch 1: smaclite import optional ────────────────────────────────────────
init_file = f'{ENVS_DIR}/__init__.py'
with open(init_file, 'r') as f:
    c = f.read()
if 'from .smaclite_wrapper import SMACliteWrapper' in c and '_smaclite_available' not in c:
    c = c.replace(
        'from .smaclite_wrapper import SMACliteWrapper',
        'try:\n    from .smaclite_wrapper import SMACliteWrapper\n'
        '    _smaclite_available = True\nexcept ImportError:\n    _smaclite_available = False'
    ).replace('REGISTRY["smaclite"] = smaclite_fn',
              'if _smaclite_available:\n    REGISTRY["smaclite"] = smaclite_fn')
    with open(init_file, 'w') as f: f.write(c)
    print('[OK] Patch 1: smaclite optional')
else:
    print('[OK] Patch 1: already applied')

# ── Patch 2: gymma.py — import tarware + fix n_agents ────────────────────────
gymma_file = f'{ENVS_DIR}/gymma.py'
with open(gymma_file, 'r') as f:
    c = f.read()
changed = False
if 'import tarware' not in c:
    c = c.replace('import gymnasium as gym',
                  'import gymnasium as gym\ntry:\n    import tarware  # noqa\nexcept ImportError:\n    pass')
    changed = True
if 'self.n_agents = self._env.unwrapped.n_agents' in c:
    c = c.replace('self.n_agents = self._env.unwrapped.n_agents',
                  '_w = self._env.unwrapped\n        self.n_agents = getattr(_w, "n_agents", None) or (getattr(_w, "num_agvs", 0) + getattr(_w, "num_pickers", 0))')
    changed = True
if changed:
    with open(gymma_file, 'w') as f: f.write(c)
    print('[OK] Patch 2: gymma.py')
else:
    print('[OK] Patch 2: already applied')

# ── Patch 3: wrappers.py — FlattenObservation.reset() for tarware API ────────
wrap_file = f'{ENVS_DIR}/wrappers.py'
with open(wrap_file, 'r') as f:
    c = f.read()
if 'tarware reset' not in c and 'obs, info = self.env.reset' in c:
    c = c.replace(
        'obs, info = self.env.reset(seed=seed, options=options)',
        'result = self.env.reset(seed=seed, options=options)\n'
        '        # tarware reset() returns tuple of obs (no info dict)\n'
        '        if isinstance(result, tuple) and len(result) == len(self.observation_space):\n'
        '            obs, info = result, {}  # tarware reset\n'
        '        else:\n'
        '            obs, info = result'
    )
    with open(wrap_file, 'w') as f: f.write(c)
    print('[OK] Patch 3: wrappers.py reset()')
else:
    print('[OK] Patch 3: already applied')

# ── Patch 4a: episode_runner.py — skip non-numeric env_info values ───────────
runner_file = f'{EPYMARL_DIR}/src/runners/episode_runner.py'
with open(runner_file, 'r') as f:
    c = f.read()
if 'isinstance(env_info.get(k, 0), (int, float))' not in c and 'env_info.get(k, 0)' in c:
    c = c.replace(
        'cur_stats.get(k, 0) + env_info.get(k, 0)',
        'cur_stats.get(k, 0) + (env_info.get(k, 0) if isinstance(env_info.get(k, 0), (int, float)) else 0)'
    )
    with open(runner_file, 'w') as f: f.write(c)
    print('[OK] Patch 4a: episode_runner.py stats')
else:
    print('[OK] Patch 4a: already applied')

# ── Patch 4b: parallel_runner.py — skip non-numeric env_info values ──────────
# Colab uses the parallel runner (batch_size_run=10, runner=parallel from ia2c.yaml).
# tarware's info dict contains 'vehicles_busy' (a list of bools) which breaks
# sum(d.get(k, 0) for d in infos) with "unsupported operand type: int + list".
par_runner_file = f'{EPYMARL_DIR}/src/runners/parallel_runner.py'
with open(par_runner_file, 'r') as f:
    c = f.read()
if 'isinstance(d.get(k, 0), (int, float))' not in c and 'sum(d.get(k, 0) for d in infos)' in c:
    c = c.replace(
        'sum(d.get(k, 0) for d in infos)',
        'sum((d.get(k, 0) if isinstance(d.get(k, 0), (int, float)) else 0) for d in infos)'
    )
    with open(par_runner_file, 'w') as f: f.write(c)
    print('[OK] Patch 4b: parallel_runner.py stats')
else:
    print('[OK] Patch 4b: already applied')

# ── Patch 5: run.py — safe datetime in save path ─────────────────────────────
run_file = f'{EPYMARL_DIR}/src/run.py'
with open(run_file, 'r') as f:
    c = f.read()
if 'strftime' not in c and 'datetime.datetime.now()' in c:
    c = c.replace('datetime.datetime.now()',
                  "datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')")
    with open(run_file, 'w') as f: f.write(c)
    print('[OK] Patch 5: run.py safe datetime')
else:
    print('[OK] Patch 5: already applied')

print(f'\nPython: {sys.executable}')
print('All installations and patches complete!')

## Step 2: Verify GPU and Environment

Check that:
1. T4 GPU is detected by PyTorch
2. The TA-RWARE environment loads correctly
3. Observation and action spaces look right for 12 agents

In [ ]:
import torch
print('=== GPU Check ===')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')
    print('Go to Runtime > Change runtime type > T4 GPU')

print('\n=== Environment Check ===')
import sys, os
# sys.path injection as fallback in case editable install hasn't propagated yet
for candidate in ['/content/tarware_repo', os.path.abspath('..')]:
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)

import gymnasium as gym
import tarware  # registers tarware envs with gymnasium

ENV_ID = 'tarware-medium-8agvs-4pickers-partialobs-v1'
env = gym.make(ENV_ID)
obs_tuple = env.reset(seed=42)  # returns tuple of N obs arrays (one per agent)

w = env.unwrapped
n_agents = w.num_agvs + w.num_pickers
print(f'Environment : {ENV_ID}')
print(f'N agents    : {n_agents} (AGVs={w.num_agvs}, Pickers={w.num_pickers})')
print(f'Max steps   : {w.max_steps}')
print(f'Reset obs   : tuple of {len(obs_tuple)} arrays, each shape {obs_tuple[0].shape}')

if hasattr(env.action_space, 'spaces'):
    n_actions = set(sp.n for sp in env.action_space.spaces)
    print(f'N actions   : {n_actions} (per agent)')
env.close()
print('\nEnvironment check PASSED!')

## Step 3: Configure IAC Training

Key hyperparameters matching the paper's setup:
- `t_max = 1,500,000` timesteps = 3,000 episodes × 500 steps/episode
- `lr = 0.0005` (learning rate for both actor and critic)
- `gamma = 0.99` (discount factor, as in paper)
- `hidden_dim = 64` (2 FC layers × 64 units, as in paper)

In [ ]:
import os, sys

EPYMARL_DIR = '/content/epymarl'
DRIVE_DIR   = '/content/drive/MyDrive/tarware_iac_results'  # for copying results to Drive
ENV_ID      = 'tarware-medium-8agvs-4pickers-partialobs-v1'
T_MAX       = 1_500_000   # ~3000 episodes * 500 steps
SEED        = 42

# EPyMARL saves results to a hardcoded path inside the epymarl directory.
# After training we copy the run directory to Drive for persistence.
SACRED_RESULTS = f'{EPYMARL_DIR}/results/sacred/ia2c/{ENV_ID}'

os.makedirs(DRIVE_DIR, exist_ok=True)

# Ensure tarware is importable (pip install -e should handle this on Colab)
for candidate in ['/content/tarware_repo']:
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)

CONFIG_OVERRIDES = [
    f'env_args.key={ENV_ID}',
    'env_args.time_limit=500',
    'common_reward=False',
    'reward_scalarisation=sum',
    # NOTE: results_save_dir is NOT a valid Sacred config key in EPyMARL.
    # Results are saved to epymarl/results/sacred/ia2c/<env>/<run_id>/ (hardcoded).
    # We copy them to Drive after training instead.
    f't_max={T_MAX}',
    f'seed={SEED}',
    'save_model=True',
    'save_model_interval=500000',
    'use_tensorboard=False',
]

print('Training Configuration:')
print(f'  algorithm : ia2c  (= IAC from paper, renamed in EPyMARL v2)')
print(f'  env       : {ENV_ID}')
print(f'  episodes  : ~{T_MAX // 500:,} (t_max={T_MAX:,} steps)')
print(f'  seed      : {SEED}')
print(f'  results   : {SACRED_RESULTS}  (auto-saved by EPyMARL)')
print(f'  drive     : {DRIVE_DIR}  (copied after training)')
print(f'\nExpected runtime: 1-2 hours on T4 GPU')

## Step 4: Run IAC Training

This cell launches EPyMARL's training loop via subprocess and streams the output in real-time.

**What to expect:**
- EPyMARL prints progress every few hundred timesteps
- Episode returns start low (random policy) and should increase over time
- Training takes ~1–2 hours on T4 GPU
- Results auto-save to Google Drive

> If the cell is interrupted, the results saved so far are still in Drive.

In [ ]:
import subprocess, sys, time, os, shutil

# ia2c = IAC from paper (renamed in this EPyMARL version)
cmd = [
    sys.executable, 'src/main.py',
    '--config=ia2c',
    '--env-config=gymma',
    'with',
] + CONFIG_OVERRIDES

# Pass tarware_repo to PYTHONPATH so the subprocess can import tarware
env = os.environ.copy()
env['PYTHONPATH'] = '/content/tarware_repo:' + env.get('PYTHONPATH', '')

print(f'Algorithm : ia2c (= IAC from paper)')
print(f'Timesteps : {T_MAX:,} (~{T_MAX//500} episodes)')
print('=' * 70)
print('Starting training... (output streamed below)')
print('=' * 70)

start_time = time.time()

try:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        cwd=EPYMARL_DIR,
        env=env,
        bufsize=1,
        universal_newlines=True,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print('\nTraining interrupted by user.')
    raise

elapsed = time.time() - start_time
print('\n' + '=' * 70)
print(f'Exit code  : {proc.returncode}')
print(f'Total time : {elapsed/3600:.2f} hours ({elapsed:.0f} seconds)')

if proc.returncode != 0:
    raise RuntimeError(
        f'EPyMARL exited with code {proc.returncode}. '
        'Fix the error above before running Steps 5-7.'
    )

# Copy Sacred results to Google Drive for persistence
import glob
run_dirs = sorted([d for d in glob.glob(f'{SACRED_RESULTS}/*/') if os.path.isdir(d)])
if run_dirs:
    latest = run_dirs[-1]
    run_id = os.path.basename(latest.rstrip('/'))
    dest = os.path.join(DRIVE_DIR, f'run_{run_id}')
    shutil.copytree(latest, dest, dirs_exist_ok=True)
    print(f'Results copied to Drive: {dest}')

print('Training completed successfully!')

## Step 5: Load Training Results

EPyMARL uses [Sacred](https://sacred.readthedocs.io) to save experiment results.
Each run creates a numbered directory with `metrics.json` containing all logged values.

In [ ]:
import json, glob, os
import numpy as np

def find_run_dirs(base_path):
    """Return numbered Sacred run directories sorted by run number, excluding _sources."""
    all_dirs = glob.glob(os.path.join(base_path, '*/'))
    return sorted(
        [d for d in all_dirs if os.path.isdir(d) and os.path.basename(d.rstrip('/\\')).isdigit()],
        key=lambda d: int(os.path.basename(d.rstrip('/\\')))
    )

# Check EPyMARL's hardcoded results path first, then Drive copy
run_dirs = find_run_dirs(SACRED_RESULTS)
if not run_dirs:
    run_dirs = find_run_dirs(os.path.join(DRIVE_DIR, 'run_1')) or \
               [d for d in glob.glob(f'{DRIVE_DIR}/run_*/') if os.path.isdir(d)]

if not run_dirs:
    print('No results found.')
    print(f'Checked: {SACRED_RESULTS}')
    print(f'Checked: {DRIVE_DIR}')
    all_in_sacred = [os.path.basename(d.rstrip('/')) for d in glob.glob(f'{SACRED_RESULTS}/*/')]
    print(f'Dirs in sacred results: {all_in_sacred}')
    print('Make sure Step 4 (training) completed before running this cell.')
else:
    LATEST_RUN = run_dirs[-1]
    run_id = os.path.basename(LATEST_RUN.rstrip('/'))
    print(f'Found {len(run_dirs)} run(s). Loading run {run_id}: {LATEST_RUN}')

    with open(os.path.join(LATEST_RUN, 'metrics.json')) as f:
        metrics = json.load(f)

    print(f'\nAvailable metrics: {list(metrics.keys())}')

    if 'return_mean' in metrics:
        data = metrics['return_mean']['values']
        n = len(data)
        print(f'\nreturn_mean: {n} data points')
        if n > 0:
            print(f'  First : step={data[0]["steps"]},   return={data[0]["value"]:.4f}')
            print(f'  Mid   : step={data[n//2]["steps"]}, return={data[n//2]["value"]:.4f}')
            print(f'  Last  : step={data[-1]["steps"]},   return={data[-1]["value"]:.4f}')

    config_path = os.path.join(LATEST_RUN, 'config.json')
    if os.path.exists(config_path):
        with open(config_path) as f:
            run_config = json.load(f)
        print(f'\nRun t_max: {run_config.get("t_max")}  seed: {run_config.get("seed")}')

    print('\nResults loaded successfully!')

## Step 6: Plot Learning Curves

We plot two views:
1. **Raw episode return** — what EPyMARL logs directly
2. **Approximate pick rate** — converted using the paper's formula

**Conversion formula:**  
`pick_rate ≈ return_sum * 3600 / (5 * 500)`  
where `return_sum = return_mean_per_agent × 12 agents`

Reference lines show how our IAC compares to:
- Our CTA heuristic (Exp 1): **52.16**
- Paper's CTA: **52.7**  
- Paper's IAC (full 10k ep): **65.2**
- Paper's HIAC (best): **66.7**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Sacred format: metrics[key] = {'steps': [...], 'values': [...]}
# NOT a list of {steps, value} dicts
METRIC_TRAIN = 'total_return_mean'
METRIC_TEST  = 'test_total_return_mean'

has_train = METRIC_TRAIN in metrics and len(metrics[METRIC_TRAIN]['values']) > 0
has_test  = METRIC_TEST  in metrics and len(metrics[METRIC_TEST]['values'])  > 0

# Convert total_return (sum of all 12 agent rewards) to approximate pick rate
# Each delivery gives +1 reward to the delivering AGV
# pick_rate = total_deliveries * 3600 / (5 * 500)
def to_pick_rate(values):
    return np.array(values) * 3600 / (5 * 500)

if not has_train and not has_test:
    print('No return metrics found. Available:', list(metrics.keys())[:10])
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('ia2c (IAC) Training -- Small GTP (tarware-medium-8agvs-4pickers)', fontsize=13)

    # Left: raw total return
    ax = axes[0]
    if has_train:
        tr_steps = np.array(metrics[METRIC_TRAIN]['steps'])
        tr_vals  = np.array(metrics[METRIC_TRAIN]['values'])
        ax.plot(tr_steps, tr_vals, alpha=0.5, color='steelblue', linewidth=1.5, label='Train return')
    if has_test:
        te_steps = np.array(metrics[METRIC_TEST]['steps'])
        te_vals  = np.array(metrics[METRIC_TEST]['values'])
        ax.plot(te_steps, te_vals, color='navy', linewidth=2.5, marker='o', markersize=5, label='Eval return')
    ax.set_xlabel('Training Timesteps')
    ax.set_ylabel('Total Episode Return (all agents)')
    ax.set_title('Raw Episode Return')
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: str(round(x/1e6, 1)) + 'M'))

    # Right: approximate pick rate with reference lines
    ax = axes[1]
    if has_train:
        ax.plot(tr_steps, to_pick_rate(tr_vals), alpha=0.4, color='steelblue', linewidth=1.5, label='Train (approx pick rate)')
    if has_test:
        ax.plot(te_steps, to_pick_rate(te_vals), color='navy', linewidth=2.5, marker='o', markersize=5, label='Eval (approx pick rate)')
    ax.axhline(y=52.16, color='darkorange', linestyle='--', linewidth=2, label='CTA ours (52.16)')
    ax.axhline(y=52.7,  color='gray',  linestyle=':', linewidth=1.5, label='CTA paper (52.7)')
    ax.axhline(y=65.2,  color='green', linestyle=':', linewidth=1.5, label='IAC paper 10k ep (65.2)')
    ax.axhline(y=66.7,  color='red',   linestyle=':', linewidth=1.5, label='HIAC paper 10k ep (66.7)')
    ax.set_xlabel('Training Timesteps')
    ax.set_ylabel('Approx. Pick Rate (order-lines / hr)')
    ax.set_title('Approximate Pick Rate vs Baselines')
    ax.legend(loc='lower right', fontsize=8); ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: str(round(x/1e6, 1)) + 'M'))

    plt.tight_layout()
    plot_path = os.path.join(DRIVE_DIR, 'iac_training_curve.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print('Plot saved to Drive:', plot_path)
    plt.show()

    final_return    = float(te_vals[-1])   if has_test  else (float(tr_vals[-1])  if has_train else None)
    final_pick_rate = float(to_pick_rate(metrics[METRIC_TEST  if has_test else METRIC_TRAIN]['values'])[-1])
    print()
    print('Final eval return    :', round(final_return, 4))
    print('Approx. pick rate    :', round(final_pick_rate, 2))

## Step 7: Performance Summary Table

Full comparison from Random baseline → CTA heuristic → IAC (ours) → paper targets.

In [ ]:
try:
    iac_pick = final_pick_rate
except NameError:
    iac_pick = None

iac_str = ('~' + str(round(float(iac_pick), 2)) + ' (approx)'
           if iac_pick is not None else '[run Step 6 first]')

print('=' * 68)
print('PERFORMANCE COMPARISON -- Small GTP (tarware-medium-8agvs-4pickers)')
print('=' * 68)
rows = [
    ('Random (our Exp 2)',          '9.63 +/- 0.12',  'ours'),
    ('CTA heuristic (our Exp 1)',   '52.16 +/- 0.54', 'ours'),
    ('CTA (paper Table I)',         '52.7 +/- 0.9',   'paper'),
    ('IAC (this notebook, 3k ep)',  iac_str,           'ours'),
    ('IAC (paper, 10k ep)',         '65.2 +/- 0.5',   'paper'),
    ('HIAC (paper, 10k ep)',        '66.7 +/- 0.3',   'paper'),
]
print('  ' + 'Method'.ljust(32) + 'Pick Rate'.rjust(16) + 'Source'.rjust(8))
print('  ' + '-' * 56)
for name, val, src in rows:
    print('  ' + name.ljust(32) + val.rjust(16) + src.rjust(8))
print('=' * 68)
print()
print('Note: IAC pick rate is approximate (converted from episode returns).')
print('      Paper trained for 10,000 episodes; we trained for ~3,000.')
print()
print('Key insight: neural RL should eventually exceed the CTA rule-based')
print('heuristic, demonstrating that learned policies beat hand-crafted rules.')

# Save summary to Drive
import json, os
summary = {
    'method': [r[0] for r in rows],
    'pick_rate_str': [r[1] for r in rows],
    'source': [r[2] for r in rows],
    'iac_pick_rate_approx': float(iac_pick) if iac_pick is not None else None,
}
out = os.path.join(DRIVE_DIR, 'iac_summary.json')
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved to Drive:', out)

---

## Notes for Report and Presentation

### What to report

1. **Algorithm**: IAC (Independent Actor-Critic) — each of 12 agents (8 AGVs + 4 pickers) maintains its own 2-layer neural network (64 units, ReLU) for both actor (policy) and critic (value estimation). Training uses GAE with γ=0.99.

2. **Framework**: EPyMARL v2 — the paper authors' own training framework, ensuring methodology is faithful to the original paper.

3. **Training setup**: 3,000 episodes (~1.5M timesteps) on Google Colab T4 GPU (~1–2 hours). The paper trained for 10,000 episodes with 5 random seeds.

4. **Result interpretation**:
   - If our IAC exceeds CTA (52.16): neural learning works, RL adds value beyond rule-based methods
   - If not yet converged: the *learning curve* itself is the deliverable — it shows the ML training is happening and heading in the right direction
   - Gap between ours (3k ep) and paper (10k ep) motivates the value of more compute

### Why IAC and not HIAC?

IAC is the simplest algorithm in the paper. HIAC adds a hierarchical manager network on top, making it significantly more complex. For a student project:
- IAC demonstrates the core RL concept clearly
- IAC results are directly comparable to paper Table I
- HIAC is the 'future work' — requires weeks more training

### Troubleshooting

- **`ModuleNotFoundError: tarware`**: Re-run the install cell
- **`No GPU detected`**: Runtime > Change runtime type > T4 GPU, then Restart runtime
- **Training very slow**: Verify GPU is active (should see 200+ FPS in EPyMARL output)
- **Sacred errors**: EPyMARL may need `pip install pymongo` for MongoDB observer
- **`env_args.key` error**: Try quoting the env ID: `with "env_args.key=tarware-medium-..."`